In [3]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive



drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/ELEYESTRA_ROUTER_V2_DATASET"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Saving to: {OUTPUT_DIR}")



FILES = [
    "batch_1_labeled.csv",
    "batch_2_combined.csv",
    "batch_3_labeled.csv",
    "batch_4_labeled.csv",
    "calendar_agent_days.csv",
    "email_prompts.csv",
    "finance_prompts.csv",
    "os_prompts.csv",
    "nan_and_unknown_labeled.csv",
    "unknown_labeled.csv",
]

# Search common Colab locations
SEARCH_DIRS = [
    "/content",
    "/content/drive/MyDrive",
]

found_files = {}

for filename in FILES:
    for directory in SEARCH_DIRS:
        path = os.path.join(directory, filename)

        if os.path.exists(path):
            found_files[filename] = path
            break




dfs = []

for filename, path in found_files.items():
    try:
        df = pd.read_csv(path)

        # Only keep datasets containing the required columns
        if "prompt" in df.columns and "category" in df.columns:
            dfs.append(df[["prompt", "category"]].copy())
            print(f"Loaded {filename}: {len(df):,} rows")
        else:
            print(f"Skipped {filename}: missing prompt/category")

    except Exception as e:
        print(f"Error loading {filename}: {e}")




combined = pd.concat(dfs, ignore_index=True)

print()
print(f"Combined before cleaning: {len(combined):,}")



combined["category"] = (
    combined["category"]
    .astype(str)
    .str.strip()
    .str.upper()
)




# Remove actual NaN values
combined = combined[
    combined["category"].notna()
].copy()

# Remove string versions of NaN
NAN_LABELS = {
    "NAN",
    "NA",
    "N/A",
    "NULL",
    "NONE",
    "",
}

before_nan = len(combined)

combined = combined[
    ~combined["category"].isin(NAN_LABELS)
].copy()

print(f"Removed NAN/empty rows: {before_nan - len(combined):,}")



UNKNOWN_LABELS = {
    "UNKNOWN",
    "UNKNOWN_CONTEXT",
}

before_unknown = len(combined)

combined = combined[
    ~combined["category"].isin(UNKNOWN_LABELS)
].copy()

print(f"Removed UNKNOWN rows: {before_unknown - len(combined):,}")




ERROR_LABELS = {
    "_ERROR_",
    "ERROR",
    "ERR",
}

before_error = len(combined)

combined = combined[
    ~combined["category"].isin(ERROR_LABELS)
].copy()

print(f"Removed ERROR rows: {before_error - len(combined):,}")




AGENT_REMAP = {
    "RESEARCH_CONTEXT": "AGENT",
    "NOTE_CONTEXT": "AGENT",
    "RESUME_CONTEXT": "AGENT",
    "EMAIL_CONTEXT": "AGENT",
    "FINANCIAL_CONTEXT": "AGENT",
    "CALENDAR_CONTEXT": "AGENT",
}

combined["category"] = combined["category"].replace(AGENT_REMAP)




combined["category"] = combined["category"].replace({
    "COMIC_CONTEXT": "PERSONAL_CONTEXT"
})




combined["prompt"] = combined["prompt"].astype(str).str.strip()

# Remove empty prompts
combined = combined[
    (combined["prompt"] != "") &
    (combined["prompt"].str.lower() != "nan")
].copy()



before_duplicates = len(combined)

combined = combined.drop_duplicates(
    subset=["prompt"],
    keep="first"
).reset_index(drop=True)

print(f"Removed duplicate prompts: {before_duplicates - len(combined):,}")



print()
print("=" * 70)
print("FINAL CATEGORY DISTRIBUTION")
print("=" * 70)

print(
    combined["category"]
    .value_counts()
    .sort_values(ascending=False)
)


# ============================================================
# 14. TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    combined,
    test_size=0.20,
    random_state=42,
    stratify=combined["category"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


# ============================================================
# 15. SAVE
# ============================================================

TRAIN_FILE = os.path.join(
    OUTPUT_DIR,
    "ELEYESTRA_ROUTER_V2_TRAIN.csv"
)

TEST_FILE = os.path.join(
    OUTPUT_DIR,
    "ELEYESTRA_ROUTER_V2_TEST.csv"
)

FULL_FILE = os.path.join(
    OUTPUT_DIR,
    "ELEYESTRA_ROUTER_V2_FULL.csv"
)

train_df.to_csv(TRAIN_FILE, index=False)
test_df.to_csv(TEST_FILE, index=False)
combined.to_csv(FULL_FILE, index=False)



print()
print("=" * 70)
print("ELEYESTRA ROUTER V2 DATASET")
print("=" * 70)

print(f"Total: {len(combined):,}")
print(f"Train: {len(train_df):,}")
print(f"Test:  {len(test_df):,}")

print()
print("TRAIN DISTRIBUTION:")
print(train_df["category"].value_counts())

print()
print("TEST DISTRIBUTION:")
print(test_df["category"].value_counts())

print()
print("=" * 70)
print("FILES SAVED")
print("=" * 70)

print(TRAIN_FILE)
print(TEST_FILE)
print(FULL_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving to: /content/drive/MyDrive/ELEYESTRA_ROUTER_V2_DATASET
Loaded batch_1_labeled.csv: 9,670 rows
Loaded batch_2_combined.csv: 4,736 rows
Loaded batch_3_labeled.csv: 9,670 rows
Loaded batch_4_labeled.csv: 9,669 rows
Loaded calendar_agent_days.csv: 3,997 rows
Loaded email_prompts.csv: 2,004 rows
Loaded finance_prompts.csv: 3,090 rows
Loaded os_prompts.csv: 1,912 rows
Loaded nan_and_unknown_labeled.csv: 3,359 rows
Loaded unknown_labeled.csv: 1,159 rows

Combined before cleaning: 49,266
Removed NAN/empty rows: 1,969
Removed UNKNOWN rows: 3,055
Removed ERROR rows: 15
Removed duplicate prompts: 1,155

FINAL CATEGORY DISTRIBUTION
category
AGENT               15629
CODING_CONTEXT      14524
GENERAL              8639
PERSONAL_CONTEXT     4280
Name: count, dtype: int64

ELEYESTRA ROUTER V2 DATASET
Total: 43,072
Train: 34,457
Test:  8,615

TRAIN DISTRIBUTION:
catego

In [4]:
!pip install transformers datasets torch scikit-learn

In [5]:
# ============================================================
# ELEYESTRA ROUTER V2
# DistilBERT 4-Class Router
# ============================================================

# ============================================================
# 1. GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import sys

# ------------------------------------------------------------
# IMPORTANT:
# Remove torchvision from the current Python process.
# This prevents Hugging Face datasets from attempting to use
# torchvision.io.VideoReader.
# ------------------------------------------------------------

for key in list(sys.modules.keys()):
    if key.startswith("torchvision"):
        del sys.modules[key]


import numpy as np
import torch

# ------------------------------------------------------------
# Disable torchvision detection inside datasets if possible.
# ------------------------------------------------------------

try:
    import datasets

    datasets.config.TORCHVISION_AVAILABLE = False

except Exception:
    pass


from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)


# ============================================================
# 3. CONFIGURATION
# ============================================================

MODEL_NAME = "distilbert-base-uncased"

BASE_DIR = "/content/drive/MyDrive/ELEYESTRA_ROUTER_V2_DATASET"

TRAIN_FILE = os.path.join(
    BASE_DIR,
    "ELEYESTRA_ROUTER_V2_TRAIN.csv"
)

TEST_FILE = os.path.join(
    BASE_DIR,
    "ELEYESTRA_ROUTER_V2_TEST.csv"
)

SAVE_DIR = (
    "/content/drive/MyDrive/"
    "elyestra_router_v2"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "elyestra_router_v2_results"
)


# ============================================================
# 4. CHECK FILES
# ============================================================

print("\n" + "=" * 70)
print("CHECKING DATASET FILES")
print("=" * 70)

if not os.path.exists(TRAIN_FILE):

    raise FileNotFoundError(
        f"\nTraining file not found:\n{TRAIN_FILE}"
    )

if not os.path.exists(TEST_FILE):

    raise FileNotFoundError(
        f"\nTest file not found:\n{TEST_FILE}"
    )

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"\n✓ Train file found:")
print(TRAIN_FILE)

print(f"\n✓ Test file found:")
print(TEST_FILE)


# ============================================================
# 5. LOAD DATASET
# ============================================================

print("\n" + "=" * 70)
print("LOADING DATASET")
print("=" * 70)

dataset = load_dataset(
    "csv",
    data_files={
        "train": TRAIN_FILE,
        "test": TEST_FILE,
    }
)

print(dataset)


# ============================================================
# 6. VERIFY DATASET COLUMNS
# ============================================================

EXPECTED_COLUMNS = {
    "prompt",
    "category",
}

for split in ["train", "test"]:

    columns = set(
        dataset[split].column_names
    )

    missing = EXPECTED_COLUMNS - columns

    if missing:

        raise ValueError(
            f"\n{split} dataset is missing columns: "
            f"{missing}\n"
            f"Found columns: {columns}"
        )


print("\n✓ Dataset schema is correct.")

print(
    "\nColumns:",
    dataset["train"].column_names
)


# ============================================================
# 7. LABEL MAPPING
# ============================================================

LABEL_MAPPING = {

    "AGENT": 0,

    "CODING_CONTEXT": 1,

    "GENERAL": 2,

    "PERSONAL_CONTEXT": 3,
}


ID2LABEL = {

    0: "AGENT",

    1: "CODING_CONTEXT",

    2: "GENERAL",

    3: "PERSONAL_CONTEXT",
}


LABEL2ID = {

    "AGENT": 0,

    "CODING_CONTEXT": 1,

    "GENERAL": 2,

    "PERSONAL_CONTEXT": 3,
}


NUM_LABELS = len(
    LABEL_MAPPING
)


# ============================================================
# 8. VERIFY LABELS
# ============================================================

print("\n" + "=" * 70)
print("VERIFYING LABELS")
print("=" * 70)

VALID_LABELS = set(
    LABEL_MAPPING.keys()
)

for split in ["train", "test"]:

    labels = set(
        dataset[split]["category"]
    )

    invalid = labels - VALID_LABELS

    if invalid:

        raise ValueError(
            f"\nInvalid labels found in {split}: "
            f"{invalid}"
        )

    print(
        f"\n✓ {split}: all labels valid"
    )


print("\nLabel mapping:")

for label, index in LABEL_MAPPING.items():

    print(
        f"  {index} → {label}"
    )


# ============================================================
# 9. CONVERT CATEGORY → INTEGER LABEL
# ============================================================

def convert_label(example):

    category = example["category"]

    return {
        "label": LABEL_MAPPING[category]
    }


dataset = dataset.map(
    convert_label
)


# ============================================================
# 10. PRINT CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("DATASET DISTRIBUTION")
print("=" * 70)

for split in ["train", "test"]:

    counts = {}

    for category in LABEL_MAPPING:

        counts[category] = (
            dataset[split]["category"]
            .count(category)
        )

    print(f"\n{split.upper()}:")

    for category, count in counts.items():

        print(
            f"{category:<20} {count:,}"
        )


# ============================================================
# 11. TOKENIZER
# ============================================================

print("\n" + "=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(
    f"\n✓ Loaded tokenizer: {MODEL_NAME}"
)


# ============================================================
# 12. TOKENIZATION
# ============================================================

def tokenize(batch):

    return tokenizer(
        batch["prompt"],

        truncation=True,

        padding="max_length",

        max_length=64,
    )


print("\nTokenizing dataset...")

dataset = dataset.map(
    tokenize,
    batched=True,
)

print("✓ Tokenization complete.")


# ============================================================
# 13. CUSTOM PYTORCH DATASET
#
# IMPORTANT:
#
# We deliberately DO NOT use:
#
# dataset.set_format(type="numpy")
#
# That was causing:
#
# torchvision.io.VideoReader
#
# ImportError.
#
# Instead, we directly read values and convert them to
# PyTorch tensors here.
# ============================================================

class QueryDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        hf_dataset
    ):

        self.data = hf_dataset

    def __len__(self):

        return len(self.data)

    def __getitem__(
        self,
        index
    ):

        row = self.data[index]

        input_ids = torch.tensor(
            row["input_ids"],
            dtype=torch.long
        )

        attention_mask = torch.tensor(
            row["attention_mask"],
            dtype=torch.long
        )

        labels = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return {

            "input_ids": input_ids,

            "attention_mask": attention_mask,

            "labels": labels,
        }


train_dataset = QueryDataset(
    dataset["train"]
)

test_dataset = QueryDataset(
    dataset["test"]
)


# ============================================================
# 14. DATASET SIZE
# ============================================================

print("\n" + "=" * 70)
print("DATASET SIZE")
print("=" * 70)

print(
    f"\nTrain examples: {len(train_dataset):,}"
)

print(
    f"Test examples:  {len(test_dataset):,}"
)


# ============================================================
# 15. TEST ONE SAMPLE
# ============================================================

print("\n" + "=" * 70)
print("TESTING DATASET")
print("=" * 70)

sample = train_dataset[0]

print(
    "\nInput IDs shape:",
    sample["input_ids"].shape
)

print(
    "Attention mask shape:",
    sample["attention_mask"].shape
)

print(
    "Label:",
    sample["labels"].item()
)

print(
    "Label name:",
    ID2LABEL[
        sample["labels"].item()
    ]
)

print(
    "\n✓ Dataset can be converted to PyTorch tensors."
)


# ============================================================
# 16. LOAD MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING MODEL")
print("=" * 70)

model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=NUM_LABELS,

    id2label=ID2LABEL,

    label2id=LABEL2ID,
)


print(
    f"\n✓ Model loaded: {MODEL_NAME}"
)

print(
    f"✓ Number of classes: {NUM_LABELS}"
)


# ============================================================
# 17. METRICS
# ============================================================

def compute_metrics(
    eval_pred
):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    macro_precision = precision_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {

        "accuracy": accuracy,

        "weighted_f1": weighted_f1,

        "macro_f1": macro_f1,

        "macro_precision": macro_precision,

        "macro_recall": macro_recall,
    }


# ============================================================
# 18. TRAINING CONFIGURATION
# ============================================================

training_args = TrainingArguments(

    output_dir=RESULTS_DIR,

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    eval_strategy="epoch",

    save_strategy="epoch",

    # --------------------------------------------------------
    # Learning rate
    # --------------------------------------------------------

    learning_rate=2e-5,

    weight_decay=0.01,

    # --------------------------------------------------------
    # Batch size
    # --------------------------------------------------------

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    # --------------------------------------------------------
    # Epochs
    # --------------------------------------------------------

    num_train_epochs=10,

    # --------------------------------------------------------
    # Best model
    # --------------------------------------------------------

    load_best_model_at_end=True,

    metric_for_best_model="eval_macro_f1",

    greater_is_better=True,

    # --------------------------------------------------------
    # Checkpoints
    # --------------------------------------------------------

    save_total_limit=2,

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    logging_steps=50,

    # --------------------------------------------------------
    # GPU
    # --------------------------------------------------------

    fp16=torch.cuda.is_available(),

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    seed=42,

    # --------------------------------------------------------
    # Disable external reporting
    # --------------------------------------------------------

    report_to="none",

    # --------------------------------------------------------
    # Keep dataset columns
    # --------------------------------------------------------

    remove_unused_columns=False,
)


# ============================================================
# 19. TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
)


# ============================================================
# 20. START TRAINING
# ============================================================

print("\n")
print("=" * 70)
print("🚀 TRAINING ELEYESTRA ROUTER V2")
print("=" * 70)

print(
    f"\nModel: {MODEL_NAME}"
)

print(
    f"Train examples: {len(train_dataset):,}"
)

print(
    f"Test examples: {len(test_dataset):,}"
)

print(
    f"Classes: {NUM_LABELS}"
)

print(
    f"GPU available: {torch.cuda.is_available()}"
)

if torch.cuda.is_available():

    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )

print("\n")


# ============================================================
# 21. TRAIN
# ============================================================

train_result = trainer.train()


# ============================================================
# 22. TRAINING SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"\nTraining loss: "
    f"{train_result.training_loss:.4f}"
)


# ============================================================
# 23. EVALUATION
# ============================================================

print("\n")
print("=" * 70)
print("📊 FINAL EVALUATION")
print("=" * 70)

results = trainer.evaluate()


for key, value in results.items():

    if isinstance(value, float):

        print(
            f"{key:<30} {value:.4f}"
        )

    else:

        print(
            f"{key:<30} {value}"
        )


# ============================================================
# 24. SAVE MODEL
# ============================================================

print("\n")
print("=" * 70)
print("💾 SAVING MODEL")
print("=" * 70)

trainer.save_model(
    SAVE_DIR
)

tokenizer.save_pretrained(
    SAVE_DIR
)

trainer.save_state()


# ============================================================
# 25. SAVE LABEL MAP
# ============================================================

label_map_file = os.path.join(
    SAVE_DIR,
    "label_mapping.txt"
)

with open(
    label_map_file,
    "w"
) as f:

    for index, label in ID2LABEL.items():

        f.write(
            f"{index}={label}\n"
        )


# ============================================================
# 26. FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("✅ ELEYESTRA ROUTER V2 TRAINING FINISHED")
print("=" * 70)

print(
    "\nModel saved to:"
)

print(
    SAVE_DIR
)

print(
    "\nLabel mapping:"
)

for index, label in ID2LABEL.items():

    print(
        f"  {index} → {label}"
    )

print(
    "\nFiles saved:"
)

print(
    f"  {SAVE_DIR}/config.json"
)

print(
    f"  {SAVE_DIR}/model.safetensors"
)

print(
    f"  {SAVE_DIR}/tokenizer.json"
)

print(
    f"  {SAVE_DIR}/label_mapping.txt"
)

print("\n✓ Done.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

CHECKING DATASET FILES

✓ Train file found:
/content/drive/MyDrive/ELEYESTRA_ROUTER_V2_DATASET/ELEYESTRA_ROUTER_V2_TRAIN.csv

✓ Test file found:
/content/drive/MyDrive/ELEYESTRA_ROUTER_V2_DATASET/ELEYESTRA_ROUTER_V2_TEST.csv

LOADING DATASET


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'category'],
        num_rows: 34457
    })
    test: Dataset({
        features: ['prompt', 'category'],
        num_rows: 8615
    })
})

✓ Dataset schema is correct.

Columns: ['prompt', 'category']

VERIFYING LABELS

✓ train: all labels valid

✓ test: all labels valid

Label mapping:
  0 → AGENT
  1 → CODING_CONTEXT
  2 → GENERAL
  3 → PERSONAL_CONTEXT


Map:   0%|          | 0/34457 [00:00<?, ? examples/s]

...



TRAINING COMPLETE

Training loss: 0.2252


📊 FINAL EVALUATION


Training Loss,Validation Loss,Epoch,Accuracy,Weighted F1,Macro F1,Macro Precision,Macro Recall
0.078888,0.885442,7,0.829019,0.828568,0.795274,0.792922,0.798826


eval_loss                      0.8854
eval_accuracy                  0.8290
eval_weighted_f1               0.8286
eval_macro_f1                  0.7953
eval_macro_precision           0.7929
eval_macro_recall              0.7988


💾 SAVING MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]



✅ ELEYESTRA ROUTER V2 TRAINING FINISHED

Model saved to:
/content/drive/MyDrive/elyestra_router_v2

Label mapping:
  0 → AGENT
  1 → CODING_CONTEXT
  2 → GENERAL
  3 → PERSONAL_CONTEXT

Files saved:
  /content/drive/MyDrive/elyestra_router_v2/config.json
  /content/drive/MyDrive/elyestra_router_v2/model.safetensors
  /content/drive/MyDrive/elyestra_router_v2/tokenizer.json
  /content/drive/MyDrive/elyestra_router_v2/label_mapping.txt

✓ Done.


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ============================================================
# CONFIG
# ============================================================

MODEL_DIR = "/content/drive/MyDrive/elyestra_router_v2"

label_mapping = {
    "AGENT": 0,
    "CODING_CONTEXT": 1,
    "GENERAL": 2,
    "PERSONAL_CONTEXT": 3,
}

# Reverse mapping: ID -> label
LABELS = {
    v: k for k, v in label_mapping.items()
}

# ============================================================
# LOAD MODEL
# ============================================================

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR
)

model.eval()

print("Model loaded successfully.")

# ============================================================
# TEST QUERIES
# ============================================================

test_queries = [

]

# ============================================================
# TEST
# ============================================================

print("\n" + "=" * 80)
print("ELEYSTRA ROUTER V2 — MANUAL TEST")
print("=" * 80)

if not test_queries:
    print("\nNo test queries added.")
    print("Add your own queries to `test_queries` and run again.\n")

else:

    for i, query in enumerate(test_queries, 1):

        # ----------------------------------------------------
        # Tokenize
        # ----------------------------------------------------

        inputs = tokenizer(
            query,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=64,
        )

        # ----------------------------------------------------
        # Inference
        # ----------------------------------------------------

        with torch.no_grad():
            outputs = model(**inputs)

        # ----------------------------------------------------
        # Probabilities
        # ----------------------------------------------------

        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )[0]

        predicted_id = torch.argmax(
            probabilities
        ).item()

        predicted_label = LABELS[predicted_id]
        confidence = probabilities[predicted_id].item()

        # ----------------------------------------------------
        # Display
        # ----------------------------------------------------

        print("\n" + "-" * 80)
        print(f"TEST #{i}")
        print("-" * 80)

        print("\nQUERY:")
        print(query)

        print("\nPREDICTION:")
        print(predicted_label)

        print("\nCONFIDENCE:")
        print(f"{confidence:.2%}")

        print("\nALL CLASS PROBABILITIES:")

        sorted_probs = sorted(
            enumerate(probabilities.tolist()),
            key=lambda x: x[1],
            reverse=True
        )

        for class_id, probability in sorted_probs:

            label = LABELS.get(
                class_id,
                f"UNKNOWN_CLASS_{class_id}"
            )

            print(
                f"  {label:<25} "
                f"{probability:.2%}"
            )

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

Loading model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully.

ELEYSTRA ROUTER V2 — MANUAL TEST

No test queries added.
Add your own queries to `test_queries` and run again.


DONE
